In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch03-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["pubchempy==1.0.5", "numpy==1.26.4", "rdkit==2026.9.1", "pandas==3.0.6", "pillow==13.0.0"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "03-01-pubchem.py"

# 查到 alanine，就能認定是手上的那瓶試劑嗎？
預設使用 2026-11-04 真實 PubChem API 快照；第三格設 LIVE=True 才會重新連線。失敗會停止，不會暗中換資料。修改後重跑第三、四格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import json
import time
import socket
from urllib.error import URLError
import pandas as pd
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Draw

# Genuine PUG REST response captured 2026-11-04; same values independently
# verified with PubChemPy 1.0.5 get_properties. Default mode uses this snapshot.
LIVE = False
PROVENANCE = {
    "retrieved": "2026-11-04",
    "name_url": "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/alanine/cids/JSON",
    "properties_url": "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/602,5950,71080/property/MolecularFormula,MolecularWeight,ConnectivitySMILES,SMILES,IUPACName,XLogP/JSON",
    "response_sha256": "10ac7535008b7178c74d0209d5790fd34f759f1f83f1931cd240f1c53e2bf4ef"
}
SNAPSHOT = [{'CID': 602, 'MolecularFormula': 'C3H7NO2', 'MolecularWeight': '89.09', 'SMILES': 'CC(C(=O)O)N', 'ConnectivitySMILES': 'CC(C(=O)O)N', 'IUPACName': '2-aminopropanoic acid', 'XLogP': -3}, {'CID': 5950, 'MolecularFormula': 'C3H7NO2', 'MolecularWeight': '89.09', 'SMILES': 'C[C@@H](C(=O)O)N', 'ConnectivitySMILES': 'CC(C(=O)O)N', 'IUPACName': '(2S)-2-aminopropanoic acid', 'XLogP': -3}, {'CID': 71080, 'MolecularFormula': 'C3H7NO2', 'MolecularWeight': '89.09', 'SMILES': 'C[C@H](C(=O)O)N', 'ConnectivitySMILES': 'CC(C(=O)O)N', 'IUPACName': '(2R)-2-aminopropanoic acid', 'XLogP': -3}]
NAME_CIDS = [5950]
CIDS = [602, 5950, 71080]
PROPERTIES = ["MolecularFormula", "MolecularWeight", "ConnectivitySMILES",
              "SMILES", "IUPACName", "XLogP"]

def request_with_retry(function, *args):
    # Retry transient transport/service errors only. Bad requests stop immediately.
    for attempt in range(3):
        try:
            return function(*args)
        except pcp.BadRequestError:
            raise
        except (pcp.ServerError, URLError, TimeoutError) as error:
            if attempt == 2:
                raise RuntimeError("PubChem 查詢失敗；請稍後重跑，未改用快照") from error
            time.sleep(2 ** (attempt + 1))

if LIVE:
    socket.setdefaulttimeout(30)
    name_cids = request_with_retry(pcp.get_cids, "alanine", "name")
    time.sleep(0.3)
    rows = request_with_retry(pcp.get_properties, PROPERTIES, CIDS, "cid")
    print("LIVE: current PubChem response")
else:
    name_cids, rows = NAME_CIDS, SNAPSHOT
    print("SNAPSHOT: retrieved", PROVENANCE["retrieved"])
print("alanine name lookup:", name_cids)
if not rows:
    raise ValueError("查詢沒有回傳性質資料")
by_cid = {row["CID"]: row for row in rows}
if len(by_cid) != len(rows) or set(by_cid) != set(CIDS):
    raise ValueError("CID 有重複或缺漏，停止對接資料")
rows = [by_cid[cid] for cid in CIDS]
required = {"CID", *PROPERTIES}
for row in rows:
    if not required <= row.keys() or any(row[k] is None for k in required):
        raise ValueError(f"欄位缺漏：{row}")
table = pd.DataFrame(rows)
table["MolecularWeight"] = pd.to_numeric(table["MolecularWeight"], errors="raise")
print(table.to_string(index=False))
mols = [Chem.MolFromSmiles(row["SMILES"]) for row in rows]
assert all(mol is not None for mol in mols)
labels = ["Unspecified stereo", "L-alanine (S)", "D-alanine (R)"]
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
image = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(330, 260),
    legends=[f"{label}\nCID {row['CID']} | MW {row['MolecularWeight']} g/mol"
             for label, row in zip(labels, rows)])
image.save(out / "03-01-pubchem.png")
print("Unique connectivity strings:", table["ConnectivitySMILES"].nunique())
print("Unique complete SMILES:", table["SMILES"].nunique())


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "03-01-pubchem.png")))